In [1]:
from dotenv import load_dotenv
load_dotenv()

import os

groq_api_key=os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq

model=ChatGroq(model="openai/gpt-oss-20b",api_key=groq_api_key)


In [3]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

document=WebBaseLoader(
    web_path="https://en.wikipedia.org/wiki/Rohit_Sharma",
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("mw-page-container"),
        )
    )
)

docs=document.load()

C:\Users\jalal\AppData\Local\Temp\ipykernel_16608\2160203254.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

split=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
chunk_docs=split.split_documents(docs)
chunk_docs

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Rohit_Sharma'}, page_content='Contents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nEarly life\n\n\n\n\n\n\n\n\n2\nYouth career\n\n\n\n\n\n\n\n\n3\nInternational career\n\n\n\n\nToggle International career subsection\n\n\n\n\n\n3.1\nTest matches\n\n\n\n\n\n\n\n\n3.2\n2015, 2019 and 2023 Cricket World Cups\n\n\n\n\n\n\n\n\n3.3\nOther one-day international matches\n\n\n\n\n\n\n\n\n3.4\nTwenty20 international matches\n\n\n\n\n\n\n\n\n\n\n4\nIndian Premier League career\n\n\n\n\n\n\n\n\n5\nPlaying style\n\n\n\n\n\n\n\n\n6\nAchievements\n\n\n\n\n\n\n\n\n7\nCommercial investments\n\n\n\n\n\n\n\n\n8\nPhilanthropy\n\n\n\n\n\n\n\n\n9\nPersonal life\n\n\n\n\nToggle Personal life subsection\n\n\n\n\n\n9.1\nCommercial endorsements\n\n\n\n\n\n\n\n\n\n\n10\nTelevision appearances\n\n\n\n\n\n\n\n\n11\nReferences\n\n\n\n\n\n\n\n\n12\nExternal links\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nToggle the table of contents\n\n\n\n\n\n\n\nRohit 

In [5]:
import os

os.environ["HUGGINGFACE_API_KEY"]=os.getenv("HUGGINGFACE_API_KEY")

from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

c:\Users\jalal\OneDrive\Desktop\Data Science\GEN AI\Langchain\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3066.51it/s]


In [6]:
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(embedding=embeddings,documents=chunk_docs)

In [7]:
retreiver=vectorstore.as_retriever(
    search_kwargs={"k":3}
)

In [8]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

contextualize_prompt=ChatPromptTemplate.from_messages(
    [
        ("system","Given the conversation history and the latest user question, "
        "rewrite the latest question as a standalone question. "
        "Do not answer the question."),

        MessagesPlaceholder(variable_name="chat_history"),

        ("human","{input}")
    ]
)

In [9]:
from langchain_classic.chains import create_history_aware_retriever

history_aware=create_history_aware_retriever(
    model,retreiver,contextualize_prompt
)

In [10]:
qa_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer the user's question using the provided context. "
        "If the answer is not present in the context, say you don't know. "
        "Keep the answer simple.\n\nContext:\n{context}"
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

In [11]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

document_chain=create_stuff_documents_chain(model,qa_prompt)
rag_chain=create_retrieval_chain(history_aware,document_chain)

In [12]:
from langchain_core.chat_history import BaseChatMessageHistory,InMemoryChatMessageHistory

store={}

def get_session_history(session_id:str)-> BaseChatMessageHistory:

    if session_id not in store:
        store[session_id]=InMemoryChatMessageHistory()

    return store[session_id]

In [13]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chatbot=RunnableWithMessageHistory(
    rag_chain,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

c:\Users\jalal\OneDrive\Desktop\Data Science\GEN AI\Langchain\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [14]:
response1=chatbot.invoke(
    {"input":"who is rohit sharma?"},
    config={
        "configurable":{
            "session_id":"chat1"
        }
    }
)

response1["answer"]

C:\Users\jalal\AppData\Local\Temp\ipykernel_16608\3861164347.py:8: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id]=InMemoryChatMessageHistory()


'Rohit\u202fSharma is an Indian international cricketer known for his batting. He has won major awards such as the ICC ODI Player of the Year 2019, led India to victories in the 2024 ICC T20 World Cup and the 2025 Champions Trophy, and received the Padma\u202fShri Award in 2026 for his contributions to cricket.'

In [15]:
response1=chatbot.invoke(
    {"input":"rohit sharma is batsman or a bowler"},
    config={
        "configurable":{
            "session_id":"chat1"
        }
    }
)

response1["answer"]

'Rohit\u202fSharma is primarily a batsman. He occasionally bowls right‑arm off‑spin but is not a regular bowler.'

In [16]:
response1=chatbot.invoke(
    {"input":"who is batsman or a bowler"},
    config={
        "configurable":{
            "session_id":"chat2"
        }
    }
)

response1["answer"]

'Rohit\u202fSharma is a **batsman** – he plays as a right‑handed top‑order batter. He does bowl right‑arm off‑break on occasion, but his main role is as a batsman.'